In [57]:
import pandas as pd
import numpy as np
import pennylane as qml
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [58]:
dataset = pd.read_csv("../dataset/riemann_features.csv")

X = dataset.drop(columns=["distance"])
y = dataset["distance"]

X = X[:2000]
y = y[:2000]

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

features = pd.read_csv("../results/data_analysis/selected_features.csv")["feature"].tolist()

# Clássico

In [59]:
def run_classical_experiment(features):
    results = []

    n_features = [i for i in range(1, 11)]

    param_grid = {
        "svr__C": [0.1, 1, 5, 10],
        "svr__epsilon": [0.001, 0.01, 0.1, 0.5],
        "svr__gamma": ["scale", 0.01, 0.1, 1.0]
    }

    for n in n_features:
        print(f"Running experiment with {n} features")

        X_train_subset = X_train[features[:n]].to_numpy()
        X_test_subset = X_test[features[:n]].to_numpy()

        pipeline = Pipeline([
            ("scaler", MinMaxScaler(feature_range=(0, np.pi))),
            ("svr", SVR(kernel="rbf"))
        ])

        grid = GridSearchCV(
            pipeline,
            param_grid,
            scoring="neg_root_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=5),
            n_jobs=-1
        )

        grid.fit(X_train_subset, y_train)

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test_subset)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "N Features": n,
            "RMSE": rmse,
            "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
            "Best gamma": grid.best_params_["svr__gamma"],
            "Features": len(features)
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


In [60]:
results = run_classical_experiment(features)
print(results)
results.to_csv("../results/experiment_14/classical_results.csv", index=False)

Running experiment with 1 features
Running experiment with 2 features
Running experiment with 3 features
Running experiment with 4 features
Running experiment with 5 features
Running experiment with 6 features
Running experiment with 7 features
Running experiment with 8 features
Running experiment with 9 features
Running experiment with 10 features
   N Features      RMSE        R2  Best C  Best epsilon Best gamma  Features
0           9  0.056229  0.961104    10.0         0.010      scale        10
1           6  0.063472  0.950439    10.0         0.010        1.0        10
2          10  0.070752  0.938417     5.0         0.010      scale        10
3           8  0.070840  0.938264    10.0         0.001      scale        10
4           5  0.075013  0.930776    10.0         0.010      scale        10
5           7  0.076022  0.928901     5.0         0.001      scale        10
6           3  0.096353  0.885787    10.0         0.010      scale        10
7           4  0.111713  0.846471

# Quantum

## PauliFeatureMap

In [61]:
def riemann_advanced_data_map(x):
    x = np.asarray(x)

    x_safe = x + 1e-8

    theta_like = np.sum(x_safe * np.log(x_safe))
    val = np.cos(theta_like)

    for i in range(len(x) - 1):
        val += 0.3 * np.cos(x[i] - x[i+1])
        val += 0.3 * np.sin(x[i] * x[i+1])

    return val


def riemann_data_map(x):
    x = np.asarray(x)
    if len(x) == 1:
        return x[0]
    x_safe = x + 1e-8
    return np.sum(x_safe * np.log(x_safe))


def create_quantum_kernel(n_qubits, reps=3, entanglement="full", paulis=["ZZ", "Z"]):
    feature_map = pauli_feature_map(
        feature_dimension=n_qubits,
        reps=reps,
        entanglement=entanglement,
        paulis=paulis,
        data_map_func=riemann_advanced_data_map    
    )
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
    return quantum_kernel


def run_quantum_experiment_pauli_feature_map():
    results = []
    n_features = [4, 6, 9, 10]

    for n in n_features:
        print(f"Running quantum experiment with {n} features using Pauli feature map")

        X_train_subset = X_train[features[:n]].to_numpy()
        X_test_subset = X_test[features[:n]].to_numpy()

        scaler_X = MinMaxScaler(feature_range=(0, np.pi))
        X_train_scaled = scaler_X.fit_transform(X_train_subset)
        X_test_scaled = scaler_X.transform(X_test_subset)

        scaler_y = MinMaxScaler(feature_range=(-1, 1))
        y_train_scaled = scaler_y.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
        y_test_scaled = scaler_y.transform(y_test.to_numpy().reshape(-1, 1)).ravel()

        quantum_kernel = create_quantum_kernel(n_qubits=n, reps=2, entanglement="full", paulis=["Y", "YY"])

        svr = SVR(kernel=quantum_kernel.evaluate, C=10, epsilon=0.01)
        svr.fit(X_train_scaled, y_train_scaled)
        y_pred = svr.predict(X_test_scaled)

        rmse = root_mean_squared_error(y_test_scaled, y_pred)
        r2 = r2_score(y_test_scaled, y_pred)

        print(f"{n} features - RMSE: {rmse:.4f}, R2: {r2:.4f}\n")

        results.append({
            "N Features": n,
            "RMSE": rmse,
            "R2": r2,
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

## Angle Encoding

In [ ]:
def make_kernel(n_qubits, rotation_first="X", rotation_second="Y"):
    dev = qml.device("lightning.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def kernel_fn(x1, x2):
        qml.AngleEmbedding(x1, wires=range(n_qubits), rotation=rotation_first)
        qml.BasicEntanglerLayers(
            weights=np.zeros((1, n_qubits)),
            wires=range(n_qubits),
            rotation=qml.RZ
        )
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation=rotation_second)
        return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))

    return kernel_fn


def make_kernel_v2(n_qubits, rotation_first="X", rotation_second="Y"):
    dev = qml.device("lightning.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def kernel_fn(x1, x2):
        qml.AngleEmbedding(x1, wires=range(n_qubits), rotation=rotation_first)
        qml.BasicEntanglerLayers(
            weights=np.zeros((1, n_qubits)),
            wires=range(n_qubits),
            rotation=qml.RZ
        )
        qml.AngleEmbedding(x2, wires=range(n_qubits), rotation=rotation_second)
        return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))

    return kernel_fn

def run_quantum_experiment_angle_encoding(features, rotation_first="X", rotation_second="Y"):
    results = []
    n_features = [4, 6, 9, 10]

    for n in n_features:
        print(f"Running quantum experiment with {n} features and rotations {rotation_first} and {rotation_second}")

        X_train_subset = X_train[features[:n]].to_numpy()
        X_test_subset = X_test[features[:n]].to_numpy()

        scaler_X = MinMaxScaler(feature_range=(0, np.pi))
        X_train_scaled = scaler_X.fit_transform(X_train_subset)
        X_test_scaled = scaler_X.transform(X_test_subset)

        scaler_y = MinMaxScaler(feature_range=(-1, 1))
        y_train_scaled = scaler_y.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
        y_test_scaled = scaler_y.transform(y_test.to_numpy().reshape(-1, 1)).ravel()

        kernel_fn = make_kernel(n, rotation_first, rotation_second)
        kernel_mat = lambda A, B, kf=kernel_fn: qml.kernels.kernel_matrix(A, B, kf)

        svr = SVR(kernel=kernel_mat, C=10, epsilon=0.01)
        svr.fit(X_train_scaled, y_train_scaled)
        y_pred = svr.predict(X_test_scaled)

        rmse = root_mean_squared_error(y_test_scaled, y_pred)
        r2 = r2_score(y_test_scaled, y_pred)

        results.append({
            "N Features": n,
            "RMSE": rmse,
            "R2": r2,
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

In [ ]:
results_pauli = run_quantum_experiment_pauli_feature_map()
print(results_pauli)
results_pauli.to_csv("../results/experiment_14/quantum_results_pauli.csv", index=False)

results_XY = run_quantum_experiment_angle_encoding(features, rotation_first="X", rotation_second="Y")
print(results_XY)
results_XY.to_csv("../results/experiment_14/quantum_results_XY.csv", index=False)

results_YY = run_quantum_experiment_angle_encoding(features, rotation_first="Y", rotation_second="Y")
print(results_YY)
results_YY.to_csv("../results/experiment_14/quantum_results_YY.csv", index=False)

Running quantum experiment with 4 features using Pauli feature map
4 features - RMSE: 3.7426, R2: -117.2667

Running quantum experiment with 6 features using Pauli feature map
6 features - RMSE: 2.4892, R2: -51.3167

Running quantum experiment with 9 features using Pauli feature map
